# 示例：酒店和航班预订代理

此解决方案将帮助您预订机票和酒店。场景是：2025年2月20日从伦敦希思罗机场(LHR)飞往纽约肯尼迪机场(JFK)，2025年2月27日返回，仅乘坐英国航空公司的经济舱。我想在纽约入住希尔顿酒店，请提供航班和酒店的费用。

# 初始化Azure AI Agent服务并从**.env**获取配置信息

### **.env** 

创建一个.env文件

**.env**包含Azure AI Agent服务的连接字符串、AOAI使用的模型以及相应的Google API搜索服务API、ENDPOINT等。

- **AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME** = "您的Azure AI Agent服务模型部署名称"

[**注意**] 您需要一个速率限制为100,000（每分钟令牌数）、速率限制为600（每分钟请求数）的模型

  您可以在Azure AI Foundry - 模型和端点中获取模型。


- **AZURE_AI_AGENT_PROJECT_CONNECTION_STRING** = "您的Azure AI Agent服务项目连接字符串"

  您可以在AI Foundry门户屏幕的项目概览中获取项目连接字符串。

- **SERPAPI_SEARCH_API_KEY** = "您的SERPAPI搜索API密钥"
- **SERPAPI_SEARCH_ENDPOINT** = "您的SERPAPI搜索端点"

要获取Azure AI Agent服务的模型部署名称和项目连接字符串，您需要创建Azure AI Agent服务。建议使用[此模板](https://portal.azure.com/#create/Microsoft.Template/uri/https%3A%2F%2Fraw.githubusercontent.com%2Ffosteramanda%2Fazure-agent-quickstart-templates%2Frefs%2Fheads%2Fmaster%2Fquickstarts%2Fmicrosoft.azure-ai-agent-service%2Fstandard-agent%2Fazuredeploy.json)直接创建（***注意：*** Azure AI Agent服务目前在有限的区域设置。建议您参考[此链接](https://learn.microsoft.com/en-us/azure/ai-services/agents/concepts/model-region-support)设置区域）

代理需要访问SERPAPI。建议使用[此链接](https://serpapi.com/searches)注册。注册后，您可以获得唯一的API密钥和端点

# 设置 

要运行此笔记本，您需要确保已通过运行`pip install -r requirements.txt`安装了所需的库。

In [ ]:
from semantic_kernel import __version__

__version__

您的Semantic Kernel版本应至少为1.27.2。

加载您的.env文件设置和资源，请确保您已添加了密钥和设置并创建了本地.env文件。

In [ ]:
from dotenv import load_dotenv

# 从.env文件加载环境变量
load_dotenv()

# 登录Azure

现在您需要登录Azure。打开终端并运行以下命令：

```bash
az login
```

此命令将提示您输入Azure凭据，使Azure AI Agent服务能够正常运行。

# 解释：
这是一个存储用于访问SERP（搜索引擎结果页面）API服务的API密钥的变量。API密钥是用于验证与您的账户关联的请求的唯一标识符。

目的：此行的目的是将API密钥存储在变量中，以便可用于验证对SERP API服务的请求。访问该服务并执行搜索需要API密钥。
如何获取SERP API密钥：要获取SERP API密钥，请按照https://serpapi.com上的这些一般步骤操作（具体步骤可能因您使用的特定SERP API服务而异）：

选择SERP API服务：有多种SERP API服务可用，如SerpAPI、Google Custom Search JSON API等。选择最适合您需求的服务。

注册账户：前往所选SERP API服务的网站并注册账户。您可能需要提供一些基本信息并验证您的电子邮件地址。

创建API密钥：注册后，登录您的账户并导航到API部分或仪表板。寻找创建或生成新API密钥的选项。
将API密钥复制到您的.env文件中。

In [ ]:
SERP_API_KEY='SERPAPI_SEARCH_API_KEY'

# 解释：
BASE_URL：这是一个存储SERP API端点基本URL的变量。变量名BASE_URL是一种约定，用于表示此URL是进行API请求的起点。
'https://serpapi.com/search'：

这是分配给BASE_URL变量的实际URL字符串。它表示使用SERP API执行搜索查询的端点。

# 目的：
此行的目的是定义一个常量，用于保存SERP API的基本URL。此URL将用作构建API请求以执行搜索操作的起点。

# 用法：
通过在变量中定义基本URL，您可以在需要向SERP API发出请求时轻松地在代码中重用它。这使您的代码更易于维护，并减少了在多个位置硬编码URL导致的错误风险。当前示例是https://serpapi.com/search?engine=bing，它使用Bing搜索API。您可以在https://Serpapi.com选择不同的API。

In [ ]:
BASE_URL = 'https://serpapi.com/search?engine=bing'

# 解释：

这是您的插件代码所在的位置。

类定义：`class BookingPlugin`：定义一个名为BookingPlugin的类，其中包含预订酒店和航班的方法。

酒店预订方法：

- `@kernel_function(description="booking hotel")`：一个装饰器，将该函数描述为用于预订酒店的内核函数。
- `def booking_hotel(self, query: Annotated[str, "The name of the city"], check_in_date: Annotated[str, "Hotel Check-in Time"], check_out_date: Annotated[str, "Hotel Check-out Time"]) -> Annotated[str, "Return the result of booking hotel information"]:`：定义一个用于预订酒店的方法，带有带注释的参数和返回类型。

该方法构建酒店预订请求的参数字典，并向SERP API发送GET请求。它检查响应状态，如果成功则返回酒店属性，如果请求失败则返回None。

航班预订方法：

- `@kernel_function(description="booking flight")`：一个装饰器，将该函数描述为用于预订航班的内核函数。
- `def booking_flight(self, origin: Annotated[str, "The name of Departure"], destination: Annotated[str, "The name of Destination"], outbound_date: Annotated[str, "The date of outbound"], return_date: Annotated[str, "The date of Return_date"]) -> Annotated[str, "Return the result of booking flight information"]:`：定义一个用于预订航班的方法，带有带注释的参数和返回类型。

该方法构建 outbound 和 return 航班请求的参数字典，并向SERP API发送GET请求。它检查响应状态，如果成功则将航班信息附加到结果字符串中，如果请求失败则打印错误消息。该方法返回包含航班信息的结果字符串。


In [ ]:
import requests

from typing import Annotated

from semantic_kernel.functions import kernel_function

# 定义预订插件
class BookingPlugin:
    """预订插件，为客户提供服务"""

    @kernel_function(description="booking hotel")
    def booking_hotel(
        self, 
        query: Annotated[str, "The name of the city"], 
        check_in_date: Annotated[str, "Hotel Check-in Time"], 
        check_out_date: Annotated[str, "Hotel Check-out Time"],
    ) -> Annotated[str, "Return the result of booking hotel information"]:
        """
        预订酒店的函数。
        参数：
        - query：城市名称
        - check_in_date：酒店入住时间
        - check_out_date：酒店退房时间
        返回：
        - 预订酒店信息的结果
        """

        # 定义酒店预订请求的参数
        params = {
            "engine": "google_hotels",
            "q": query,
            "check_in_date": check_in_date,
            "check_out_date": check_out_date,
            "adults": "1",
            "currency": "GBP",
            "gl": "uk",
            "hl": "en",
            "api_key": SERP_API_KEY
        }

        # 向SERP API发送GET请求
        response = requests.get(BASE_URL, params=params)

        # 检查请求是否成功
        if response.status_code == 200:
            # 将响应内容解析为JSON
            response = response.json()
            # 返回响应中的属性
            return response["properties"]
        else:
            # 如果请求失败，返回None
            return None

    @kernel_function(description="booking flight")
    def booking_flight(
        self, 
        origin: Annotated[str, "The name of Departure"], 
        destination: Annotated[str, "The name of Destination"], 
        outbound_date: Annotated[str, "The date of outbound"], 
        return_date: Annotated[str, "The date of Return_date"],
    ) -> Annotated[str, "Return the result of booking flight information"]:
        """
        预订航班的函数。
        参数：
        - origin：出发地名称
        - destination：目的地名称
        - outbound_date：出发日期
        - return_date：返回日期
        - airline：首选航空公司
        - hotel_brand：首选酒店品牌
        返回：
        - 预订航班信息的结果
        """
        
        # 定义出发航班请求的参数
        go_params = {
            "engine": "google_flights",
            "departure_id": "destination",
            "arrival_id": "origin",
            "outbound_date": "outbound_date",
            "return_date": "return_date",
            "currency": "GBP",
            "hl": "en",
            "airline": "airline",
            "hotel_brand": "hotel_brand",
            "api_key": "SERP_API_KEY"
        }
         print(go_params)

        # 发送出发航班的GET请求
        go_response = requests.get(BASE_URL, params=go_params)

        # 初始化结果字符串
        result = ''

        # 检查出发航班请求是否成功
        if go_response.status_code == 200:
            # 将响应内容解析为JSON
            response = go_response.json()
            # 将出发航班信息附加到结果中
            result += "# outbound \n " + str(response)
        else:
            # 如果请求失败，打印错误消息
            print('error!!!')

        # 定义返回航班请求的参数
        back_params = {
            #"engine": "google_flights",
            "departure_id": destination,
            "arrival_id": origin,
            "outbound_date": outbound_date,
            "return_date": return_date,
            "currency": "GBP",
            "hl": "en",
            "api_key": SERP_API_KEY
        }

        # 发送返回航班的GET请求
        back_response = requests.get(BASE_URL, params=back_params)

        # 检查返回航班请求是否成功
        if back_response.status_code == 200:
            # 将响应内容解析为JSON
            response = back_response.json()
            # 将返回航班信息附加到结果中
            result += "\n # return \n" + str(response)
        else:
            # 如果请求失败，打印错误消息
            print('error!!!')

        # 打印结果
        print(result)

        # 返回结果
        return result


# 解释：
导入语句：导入用于Azure凭据、AI代理、聊天消息内容、作者角色和内核函数装饰器的必要模块。

异步上下文管理器：async with (DefaultAzureCredential() as creds, AzureAIAgent.create_client(credential=creds, conn_str="...") as client,): 这设置了一个异步上下文管理器来处理Azure凭据并创建AI代理客户端。

代理名称和指令：
- `AGENT_NAME = "BookingAgent"`：定义代理的名称。
- `AGENT_INSTRUCTIONS = """..."""`：为代理提供有关如何处理预订请求的详细指令。

创建代理定义：`agent_definition = await client.agents.create_agent(...)`：使用指定的模型、名称和指令创建代理定义。

创建AzureAI代理：`agent = AzureAIAgent(...)`：使用客户端、代理定义和定义的插件创建AzureAI代理。

创建线程：`thread: AzureAIAgentThread | None = None`：为代理创建一个线程。不需要先创建线程 - 如果提供`None`值，将在第一次调用期间创建一个新线程，并作为响应的一部分返回。

用户输入：`user_inputs = ["..."]`：定义代理要处理的用户输入列表。

在finally块中，删除线程和代理以清理资源。

# 身份验证

`DefaultAzureCredential`类是Azure SDK for Python的一部分。它提供了一种默认方式来向Azure服务进行身份验证。它尝试使用多种方法按特定顺序进行身份验证，例如环境变量、托管标识和Azure CLI凭据。

异步操作：aio模块表示DefaultAzureCredential类支持异步操作。这意味着您可以将其与asyncio一起使用来执行非阻塞身份验证请求。

In [ ]:
# 导入必要的模块
from azure.identity.aio import DefaultAzureCredential
from semantic_kernel.agents import AzureAIAgent, AzureAIAgentSettings, AzureAIAgentThread

ai_agent_settings = AzureAIAgentSettings.create()

# Azure AI设置
async with (
     DefaultAzureCredential() as creds,
    AzureAIAgent.create_client(
        credential=creds,
        conn_str=ai_agent_settings.project_connection_string.get_secret_value(),
    ) as client,
):    
    
    # 定义代理的名称和指令
    AGENT_NAME = "BookingAgent"
    AGENT_INSTRUCTIONS = """
    您是一名预订代理，帮助我预订航班或酒店。

    思考：理解用户的意图并确认是否使用预订系统完成任务。

    行动：
    - 如果预订航班，将出发地名称和目的地名称转换为机场代码。
    - 如果预订酒店或航班，使用相应的API进行调用。确保必要的参数可用。如果缺少任何参数，使用默认值或假设继续进行。
    - 如果不是酒店或航班预订，仅使用最终答案进行响应。
    - 使用markdown表格输出结果：
    - 对于航班预订，将出发和返回内容分开，并按照以下顺序列出：出发机场名称 | 航空公司 | 航班号 | 出发时间 | 到达机场名称 | 到达时间 | 持续时间 | 飞机 | 旅行舱位 | 价格（USD） | 腿部空间 | 扩展服务 | 碳排放（kg）。
    - 对于酒店预订，按照以下顺序列出：酒店名称 | 酒店描述 | 入住时间 | 退房时间 | 价格 | 附近地点 | 酒店等级 | GPS坐标。
    """

    # 使用指定的模型、名称和指令创建代理定义
    agent_definition = await client.agents.create_agent(
        model=ai_agent_settings.model_deployment_name,
        name=AGENT_NAME,
        instructions=AGENT_INSTRUCTIONS,
    )

    # 使用客户端和代理定义创建AzureAI代理
    agent = AzureAIAgent(
        client=client,
        definition=agent_definition,
        plugins=[BookingPlugin()]
    )

    # 为代理创建一个新线程
    # 如果未提供线程，将
    # 创建一个新线程并与初始响应一起返回
    thread: AzureAIAgentThread | None = None

    # 这是您要完成的活动或任务的提示
    # 定义代理要处理的用户输入，我们提供了一些示例提示来测试和验证
    user_inputs = [
        # "Can you tell me the round-trip air ticket from  London to New York JFK aiport, the departure time is February 17, 2025, and the return time is February 23, 2025"
        # "Book a hotel in New York from Feb 20,2025 to Feb 24,2025"
        "Help me book flight tickets and hotel for the following trip London Heathrow LHR Feb 20th 2025 to New York JFK returning Feb 27th 2025 flying economy with British Airways only. I want a stay in a Hilton hotel in New York please provide costs for the flight and hotel"
        # "I have a business trip from London LHR to New York JFK on Feb 20th 2025 to Feb 27th 2025, can you help me to book a hotel and flight tickets"
    ]

    try:
        # 处理每个用户输入
        for user_input in user_inputs:
            print(f"# User: '{user_input}'")
            # 获取代理对指定线程的响应
            response = await agent.get_response(
                messages=user_input,
                thread=thread,
            )
            thread = response.thread
            # 打印代理的响应
            print(f"{response.name}: '{response.content}'")
    finally:
        # 通过删除线程和代理进行清理
        await thread.delete() if thread else None
        await client.agents.delete_agent(agent.id)